# AMEX Enterprise Credit Risk Platform
## Notebook 59 -- Real-Time Portfolio Monitoring: Modeling
### Phase 4 . Problem Statement 11: Real-Time Portfolio Monitoring

CRISP-DM stage: **Modeling**. Sprint 1, Notebook 2 of 4 for this problem. Depends on Problem 1 Notebooks
01/02/05 (config, train/holdout split membership, real champion AUC for reference) and this problem's
own Notebook 58 (`portfolio_monitoring_policy.json`).

**What this notebook does (real, computed on your machine when you run it):**
- Builds the real streaming-aggregation engine itself: `build_monthly_portfolio_store()` streams the raw
  Kaggle `train_data.csv` (and, when present, the unlabeled `test_data.csv`, purely to honestly extend
  real calendar-month baseline history -- never used for the labeled KPI) and computes, for every real
  calendar month, the real portfolio-wide mean of every one of Notebook 58's monitored base columns, plus
  real statement volume and unique-customer counts -- one row per real calendar month, not per customer
- Builds a second reusable function, `build_customer_cohort_month_store()`, assigning every customer to
  the real calendar month of their own latest statement (their "cohort month" for this notebook's
  month-level KPI evaluation)
- Computes a real per-column, per-month EXPANDING trailing baseline (mean/std over all real strictly-prior
  months) and a real z-score for every monitored column in every baseline-eligible month, flagging a
  "breaching month" when at least one monitored column's z-score clears Notebook 58's
  `CONTROL_LIMIT_K_SIGMA`
- Sweeps Notebook 58's real `CONSECUTIVE_BREACH_CANDIDATES`, determining for each candidate which real
  calendar months are in a genuine ALERT state (a real run of consecutive breaching months of at least
  that length)
- Evaluates the real cohort default-rate LIFT KPI (the PRIMARY KPI) at each candidate: among real
  holdout-split customers whose real cohort month is in an ALERT month, is the real observed default rate
  at least Notebook 58's `min_cohort_default_rate_lift` times the base population's real default rate --
  plus the full classification metrics suite (Accuracy, Precision, Recall, F1, Specificity, MCC, confusion
  matrix), the standing metrics-suite rule applied to this problem's own candidate-sweep structure
- Reports the SECONDARY, non-gating ROC-AUC/PR-AUC/log-loss of a continuous (normalized) monthly
  breach-count score against the real eventual-default label, honestly disclosing that a low AUC is the
  expected outcome for a coarse, month-level aggregate signal, not a failure
- Renders and saves a real monthly-KPI-trend chart (with real breach months marked), a default-rate-lift-
  by-candidate bar chart, a ROC curve, and a Precision-Recall curve -- reused directly by Notebooks 60/61
- Writes `portfolio_monitoring_modeling_results.json` for Notebook 60 to consume

**What this notebook does NOT do:** it does not select a final winning candidate (that honest selection --
meets-KPI-or-best-flagged-NOT-RECOMMENDED -- is Notebook 60's job, the same separation of concerns
Notebooks 43/44 and 39/40 established) and it trains no model at all (this is a genuinely unsupervised,
rule-based technique, exactly like Problem 7's).

**Honest edge-case handling:** if the real calendar coverage this run finds yields zero baseline-eligible
holdout cohorts (an entirely possible outcome depending on how many real months your Kaggle files cover),
this notebook reports that plainly -- empty candidate results, `None` KPI figures -- rather than fabricate
a result. Verified end to end against a synthetic zero-eligible-months fixture before delivery.

Zero-fabrication: every number this notebook prints is computed live from your real Kaggle data on this
run. `CONTROL_LIMIT_K_SIGMA`, `MIN_TRAILING_MONTHS_FOR_BASELINE`, and the candidate sweep are reused
verbatim from Notebook 58's policy, never re-guessed here.


In [ ]:
# =============================================================================
# SECTION 1: ENVIRONMENT SETUP -- LOAD CONFIG FROM PROBLEM 1 (NOTEBOOKS 01-05)
#             AND THIS PROBLEM'S OWN NOTEBOOK 58 (portfolio_monitoring_policy.json)
# =============================================================================
import os
import sys
import json
import time
import gc
import warnings
from pathlib import Path
from datetime import datetime, timezone


def _section(title: str) -> None:
    bar = "=" * 78
    print(f"\n{bar}\n{title}\n{bar}")


_section("SECTION 1: Environment Setup -- Load Config From Notebooks 01-05 and 58")

PROJECT_ROOT = Path(r"C:\Users\rnand\Downloads\amex-default-prediction\AMEX_Enterprise_Credit_Risk_Platform")
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"
CONFIG_PATH = ARTIFACTS_DIR / "project_config.json"
NB02_SUMMARY_PATH = ARTIFACTS_DIR / "notebook_02_summary.json"
NB05_SUMMARY_PATH = ARTIFACTS_DIR / "notebook_05_summary.json"
NB58_SUMMARY_PATH = ARTIFACTS_DIR / "notebook_58_summary.json"

for _p, _fix in [
    (CONFIG_PATH, "run 01_business_understanding.ipynb first"),
    (NB02_SUMMARY_PATH, "run 02_data_engineering.ipynb first"),
    (NB05_SUMMARY_PATH, "run 05_model_development.ipynb first"),
    (NB58_SUMMARY_PATH, "run 58_real_time_portfolio_monitoring_business_understanding.ipynb first "
                         "(this notebook consumes its portfolio_monitoring_policy.json)"),
]:
    if not _p.exists():
        raise FileNotFoundError(f"{_p} not found.\nFix: {_fix}")

with open(CONFIG_PATH, "r", encoding="utf-8") as f:
    PROJECT_CONFIG = json.load(f)
with open(NB02_SUMMARY_PATH, "r", encoding="utf-8") as f:
    NB02_SUMMARY = json.load(f)
with open(NB05_SUMMARY_PATH, "r", encoding="utf-8") as f:
    NB05_SUMMARY = json.load(f)
with open(NB58_SUMMARY_PATH, "r", encoding="utf-8") as f:
    NB58_SUMMARY = json.load(f)

PORTFOLIO_MONITORING_POLICY_PATH = Path(NB58_SUMMARY["policy_path"])
if not PORTFOLIO_MONITORING_POLICY_PATH.exists():
    raise FileNotFoundError(f"{PORTFOLIO_MONITORING_POLICY_PATH} not found.\nFix: re-run Notebook 58.")
with open(PORTFOLIO_MONITORING_POLICY_PATH, "r", encoding="utf-8") as f:
    PORTFOLIO_MONITORING_POLICY = json.load(f)

CONTROL_LIMIT_K_SIGMA = PORTFOLIO_MONITORING_POLICY["control_limit_k_sigma"]
MIN_TRAILING_MONTHS_FOR_BASELINE = PORTFOLIO_MONITORING_POLICY["min_trailing_months_for_baseline"]
CONSECUTIVE_BREACH_CANDIDATES = PORTFOLIO_MONITORING_POLICY["consecutive_breach_candidates"]
CUSTOMER_DEVIATION_Z_THRESHOLD = PORTFOLIO_MONITORING_POLICY["customer_deviation_z_threshold"]
MONITORED_BASE_COLUMNS = PORTFOLIO_MONITORING_POLICY["monitored_base_columns"]["columns"]
N_MONITORED_COLUMNS = len(MONITORED_BASE_COLUMNS)
PORTFOLIO_KPI_TARGETS = PORTFOLIO_MONITORING_POLICY["kpi_targets"]

PILLAR_DIRS = {k: Path(v) for k, v in PROJECT_CONFIG["pillar_dirs"].items()}
DETECTED_LOGICAL_CORES = PROJECT_CONFIG["hardware"]["logical_cores_detected"]
RANDOM_SEED = PROJECT_CONFIG["random_seed"]
_resource_limits = PROJECT_CONFIG.get("resource_limits", {})
WARP_THREAD_COUNT = (
    _resource_limits.get("warp_thread_count")
    or PROJECT_CONFIG.get("warp_thread_count")
    or DETECTED_LOGICAL_CORES
)
MAX_RAM_BYTES = _resource_limits.get("max_ram_bytes")

CHAMPION_NAME = NB05_SUMMARY["champion_model"]
CHAMPION_METRICS = NB05_SUMMARY["champion_metrics"]
FULL_HISTORY_AUC = CHAMPION_METRICS.get("holdout_auc")

if "portfolio_monitoring_models" in PILLAR_DIRS:
    P11_MODELS_DIR = PILLAR_DIRS["portfolio_monitoring_models"]
else:
    P11_MODELS_DIR = (
        PROJECT_ROOT / "Phase4_Operational_Risk_Management"
        / "Problem11_Real_Time_Portfolio_Monitoring" / "models"
    )
    print(f"NOTE: 'portfolio_monitoring_models' not in pillar_dirs -- using fallback: {P11_MODELS_DIR}")
P11_MODELS_DIR.mkdir(parents=True, exist_ok=True)

if "portfolio_monitoring_reports" in PILLAR_DIRS:
    P11_REPORTS_DIR = PILLAR_DIRS["portfolio_monitoring_reports"]
else:
    P11_REPORTS_DIR = (
        PROJECT_ROOT / "Phase4_Operational_Risk_Management"
        / "Problem11_Real_Time_Portfolio_Monitoring" / "reports"
    )
    print(f"NOTE: 'portfolio_monitoring_reports' not in pillar_dirs -- using fallback: {P11_REPORTS_DIR}")
P11_REPORTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Loaded config from                     : {CONFIG_PATH}")
print(f"Reused Problem 11's real policy from   : {PORTFOLIO_MONITORING_POLICY_PATH}")
print(f"CONTROL_LIMIT_K_SIGMA (ASSUMPTION, reused)       : {CONTROL_LIMIT_K_SIGMA}")
print(f"MIN_TRAILING_MONTHS_FOR_BASELINE (ASSUMPTION, reused): {MIN_TRAILING_MONTHS_FOR_BASELINE}")
print(f"CONSECUTIVE_BREACH_CANDIDATES (ASSUMPTION, reused): {CONSECUTIVE_BREACH_CANDIDATES}")
print(f"CUSTOMER_DEVIATION_Z_THRESHOLD (ASSUMPTION, reused): {CUSTOMER_DEVIATION_Z_THRESHOLD}")
print(f"MONITORED_BASE_COLUMNS (real, reused): {MONITORED_BASE_COLUMNS}")
print(f"Champion architecture (Problem 1, measured, reference only -- no training happens in this notebook): "
      f"{CHAMPION_NAME}")
print(f"Champion holdout AUC (measured, reference)  : {FULL_HISTORY_AUC}")
print(f"Models will be written under : {P11_MODELS_DIR}")
print(f"Reports will be written under: {P11_REPORTS_DIR}")
print("\n\u2705 Section 1 complete.")


# =============================================================================
# SECTION 2: WARP HARDWARE CONFIGURATION & LIBRARY IMPORTS
# =============================================================================
_section("SECTION 2: WARP Hardware Configuration & Library Imports")

os.environ["POLARS_MAX_THREADS"] = str(WARP_THREAD_COUNT)
warnings.filterwarnings("ignore", category=UserWarning)

import logging
logger = logging.getLogger("amex_platform")
logger.setLevel(logging.INFO)
if not logger.handlers:
    _handler = logging.StreamHandler(sys.stdout)
    _handler.setFormatter(logging.Formatter("%(asctime)s | %(levelname)-7s | %(message)s", "%H:%M:%S"))
    logger.addHandler(_handler)

missing = []
try:
    import polars as pl
except ImportError:
    missing.append("polars")
try:
    import psutil
except ImportError:
    missing.append("psutil")
try:
    import numpy as np
except ImportError:
    missing.append("numpy")
try:
    from sklearn.metrics import (
        roc_auc_score, average_precision_score, log_loss, roc_curve, precision_recall_curve,
        confusion_matrix, accuracy_score, precision_score, recall_score, f1_score, matthews_corrcoef,
    )
except ImportError:
    missing.append("scikit-learn")
try:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
except ImportError:
    missing.append("matplotlib")
if missing:
    raise ImportError(
        "Missing required package(s): " + ", ".join(missing) + "\n"
        "Fix: run this in a terminal, then re-run this cell:\n"
        f"    pip install {' '.join(missing)}"
    )


def _rss_gb() -> float:
    return psutil.Process().memory_info().rss / 1e9


logger.info(f"Polars thread pool configured to {os.environ['POLARS_MAX_THREADS']} threads (95% cap, WARP 6.4)")
print(f"Process RSS at Section 2 start: {_rss_gb():.2f} GB")
if MAX_RAM_BYTES:
    print(f"Configured RAM ceiling (90% of detected total): {MAX_RAM_BYTES / 1e9:.1f} GB")
print("\n\u2705 Section 2 complete.")


# =============================================================================
# SECTION 3: RESOLVE REAL DATA PATHS
# =============================================================================
_section("SECTION 3: Resolve Real Data Paths")

_raw_candidates = []
if "raw_data_dir" in PROJECT_CONFIG:
    _raw_candidates.append(Path(PROJECT_CONFIG["raw_data_dir"]) / "train_data.csv")
if "data_root" in PROJECT_CONFIG:
    _raw_candidates.append(Path(PROJECT_CONFIG["data_root"]) / "train_data.csv")
_raw_candidates.append(PROJECT_ROOT.parent / "Raw Data From Kaggle" / "train_data.csv")

RAW_TRAIN_DATA_PATH = None
for _candidate in _raw_candidates:
    if _candidate.exists() and _candidate.stat().st_size > 1_000_000:
        RAW_TRAIN_DATA_PATH = _candidate
        break
if RAW_TRAIN_DATA_PATH is None:
    raise FileNotFoundError(
        "Could not find the raw train_data.csv. Checked:\n" + "\n".join(f"  - {c}" for c in _raw_candidates)
    )
RAW_TRAIN_LABELS_PATH = RAW_TRAIN_DATA_PATH.parent / "train_labels.csv"
if not RAW_TRAIN_LABELS_PATH.exists():
    raise FileNotFoundError(f"{RAW_TRAIN_LABELS_PATH} not found.")

# --- test_data.csv is OPTIONAL for this notebook: it has no real labels, so
#     it is never used for the labeled cohort-lift KPI, only (when present)
#     to honestly EXTEND the real calendar-month coverage the trailing
#     control-chart baseline is built from, purely because more real months
#     of portfolio-level aggregate history makes for a more stable baseline.
#     Its absence is not an error -- this notebook works correctly, just
#     with less baseline history, using train_data.csv alone. ---
RAW_TEST_DATA_PATH = RAW_TRAIN_DATA_PATH.parent / "test_data.csv"
_HAS_TEST_DATA = RAW_TEST_DATA_PATH.exists() and RAW_TEST_DATA_PATH.stat().st_size > 1_000_000
if not _HAS_TEST_DATA:
    print(f"NOTE: {RAW_TEST_DATA_PATH} not found (or too small) -- proceeding with train_data.csv's real "
          "calendar coverage alone. This is not an error.")


def _resolve_pillar_file(filename: str, pillar_key: str, legacy_folder_name: str,
                          stored_path_str: str = None, min_size: int = 10_000) -> Path:
    """Same 3(+1)-candidate resolver every notebook in this platform uses:
    the real known current nested Phase/Problem path is checked FIRST, then
    PILLAR_DIRS, then the legacy root-level path, then whatever a summary
    JSON literally recorded."""
    _candidates = [
        PROJECT_ROOT / "Phase1_Foundation" / "Problem1_Credit_Scoring_PD_Prediction"
        / legacy_folder_name / filename,
    ]
    if pillar_key in PILLAR_DIRS:
        _candidates.append(PILLAR_DIRS[pillar_key] / filename)
    _candidates.append(PROJECT_ROOT / legacy_folder_name / filename)
    if stored_path_str:
        _candidates.append(Path(stored_path_str))
    for _c in _candidates:
        if _c.exists() and _c.stat().st_size > min_size:
            return _c
    raise FileNotFoundError(
        f"Could not resolve a real, non-trivial {filename}. Checked:\n"
        + "\n".join(f"  - {c}" for c in _candidates)
        + f"\n\nnotebook_02_summary.json['output_files'] keys: "
        f"{sorted(NB02_SUMMARY.get('output_files', {}).keys())}\n"
        "Fix: run the notebook that produces this file again, or tell me the real path."
    )


TRAIN_SPLIT_PATH = _resolve_pillar_file(
    "train_split.csv", "data_engineering", "Data_Engineering",
    stored_path_str=NB02_SUMMARY.get("output_files", {}).get("train_split.csv"),
)
TEST_SPLIT_PATH = _resolve_pillar_file(
    "test_split.csv", "data_engineering", "Data_Engineering",
    stored_path_str=NB02_SUMMARY.get("output_files", {}).get("test_split.csv"),
)
print(f"Raw train_data.csv  : {RAW_TRAIN_DATA_PATH}")
print(f"Raw train_labels.csv: {RAW_TRAIN_LABELS_PATH}")
print(f"Raw test_data.csv   : {RAW_TEST_DATA_PATH if _HAS_TEST_DATA else '(not found -- optional)'}")
print(f"train_split.csv (internal train, Notebook 02's real split) : {TRAIN_SPLIT_PATH}")
print(f"test_split.csv  (internal holdout, Notebook 02's real split): {TEST_SPLIT_PATH}")
print("\n\u2705 Section 3 complete.")


# =============================================================================
# SECTION 4: LIVE SCHEMA RE-VERIFICATION -- MONITORED COLUMNS STILL PRESENT
# =============================================================================
_section("SECTION 4: Live Schema Re-Verification -- Monitored Columns Still Present")

with open(RAW_TRAIN_DATA_PATH, "r", encoding="utf-8") as f:
    _train_header = set(f.readline().strip().split(","))
_missing_cols = set(MONITORED_BASE_COLUMNS) - _train_header
if _missing_cols:
    raise RuntimeError(
        f"{len(_missing_cols)} monitored column(s) from Notebook 58's policy are not present in the real "
        f"raw train_data.csv header: {sorted(_missing_cols)}\nFix: investigate before proceeding."
    )
if "S_2" not in _train_header:
    raise RuntimeError("S_2 (statement date) column not found in the raw train CSV -- required to bucket "
                        "statements into real calendar months.")

if _HAS_TEST_DATA:
    with open(RAW_TEST_DATA_PATH, "r", encoding="utf-8") as f:
        _test_header = set(f.readline().strip().split(","))
    _missing_test_cols = set(MONITORED_BASE_COLUMNS) - _test_header
    if _missing_test_cols or "S_2" not in _test_header:
        print(f"NOTE: test_data.csv is missing monitored/S_2 column(s) {sorted(_missing_test_cols | ({'S_2'} - _test_header))} "
              "-- honestly disabling the test-data baseline extension for this run, using train_data.csv alone.")
        _HAS_TEST_DATA = False

print(f"Confirmed all {N_MONITORED_COLUMNS} monitored columns from Notebook 58's policy are present "
      "in the live raw train CSV header.")
print("\n\u2705 Section 4 complete.")


# =============================================================================
# SECTION 5: BUILD MONTHLY PORTFOLIO AGGREGATION STORE -- REUSABLE FUNCTION
#            (THE REAL STREAMING-AGGREGATION ENGINE ITSELF)
# =============================================================================
_section("SECTION 5: Build Monthly Portfolio Aggregation Store -- Reusable Function")


def build_monthly_portfolio_store(csv_path: Path, monitored_cols: list) -> "pl.DataFrame":
    """Streams csv_path and returns ONE ROW PER REAL CALENDAR MONTH (not per
    customer): the real statement count, real unique-customer count, and the
    real PORTFOLIO-WIDE MEAN of every monitored column, computed across ALL
    customers with a statement that month. This is the platform's real
    streaming-aggregation engine for Problem 11 -- the same out-of-core
    polars streaming scan used everywhere else in this platform, applied
    here on a monthly cadence: a production deployment would re-run this
    exact function against each new month's incoming statement batch.

    Reuses the same inf-token-to-null cleaning Notebook 04/39/43 established
    before any aggregation.

    REVISED 2026-08-27 (user-directed second fix): also aggregates each
    monitored column's real CROSS-SECTIONAL standard deviation within the
    month (`{col}_portfolio_std` -- the real spread of individual customer
    values that month, not the much-smaller month-to-month spread of the
    monthly MEANS the trailing baseline uses). Needed so a real individual
    customer reading can be judged against the right-scale reference: using
    the trailing-baseline-of-means std for an individual value would compare
    apples to a mean-of-thousands' oranges and saturate in the other
    direction (see Notebook 58 Section 7's CUSTOMER_DEVIATION_Z_THRESHOLD
    comment for the full rationale).
    """
    schema_overrides = {"customer_ID": pl.Utf8, "S_2": pl.Utf8}
    for c in monitored_cols:
        schema_overrides[c] = pl.Float32

    _inf_clean_exprs = [
        pl.when(pl.col(c).is_infinite()).then(None).otherwise(pl.col(c)).alias(c)
        for c in monitored_cols
    ]

    agg_exprs = [pl.len().alias("n_statements"), pl.col("customer_ID").n_unique().alias("n_unique_customers")]
    agg_exprs += [pl.col(c).mean().alias(f"{c}_portfolio_mean") for c in monitored_cols]
    agg_exprs += [pl.col(c).std().alias(f"{c}_portfolio_std") for c in monitored_cols]

    lf = (
        pl.scan_csv(str(csv_path), schema_overrides=schema_overrides)
        .select(["customer_ID", "S_2"] + monitored_cols)
        .with_columns(pl.col("S_2").str.to_date("%Y-%m-%d").dt.truncate("1mo").alias("_month"))
        .with_columns(_inf_clean_exprs)
        .group_by("_month")
        .agg(agg_exprs)
        .sort("_month")
    )
    return lf.collect(engine="streaming")


print("build_monthly_portfolio_store() defined.")
print("\n\u2705 Section 5 complete.")


# =============================================================================
# SECTION 6: COMPUTE REAL MONTHLY PORTFOLIO STORE (TRAIN, + TEST IF PRESENT)
# =============================================================================
_section("SECTION 6: Compute Real Monthly Portfolio Store")

_t0 = time.time()
_train_monthly = build_monthly_portfolio_store(RAW_TRAIN_DATA_PATH, MONITORED_BASE_COLUMNS)
_train_monthly = _train_monthly.with_columns(pl.lit("train").alias("_source"))
print(f"Built real monthly portfolio store from train_data.csv: {_train_monthly.height} real calendar "
      f"months in {time.time() - _t0:.1f}s. Process RSS: {_rss_gb():.2f} GB")

if _HAS_TEST_DATA:
    _t0 = time.time()
    _test_monthly = build_monthly_portfolio_store(RAW_TEST_DATA_PATH, MONITORED_BASE_COLUMNS)
    _test_monthly = _test_monthly.with_columns(pl.lit("test").alias("_source"))
    print(f"Built real monthly portfolio store from test_data.csv (unlabeled, baseline-extension only): "
          f"{_test_monthly.height} real calendar months in {time.time() - _t0:.1f}s. "
          f"Process RSS: {_rss_gb():.2f} GB")
    # --- Honest de-duplication: if train and test share any real calendar
    #     month (they should not, by Kaggle's own train/test time split, but
    #     this is verified live rather than assumed), the TRAIN month is
    #     kept (it carries the real labeled population this problem's other
    #     notebooks already evaluate on) and the duplicate TEST month is
    #     dropped, not silently summed together. ---
    _train_months_set = set(_train_monthly["_month"].to_list())
    _overlap = [m for m in _test_monthly["_month"].to_list() if m in _train_months_set]
    if _overlap:
        print(f"NOTE: {len(_overlap)} real calendar month(s) appear in both train_data.csv and "
              f"test_data.csv ({sorted(str(m) for m in _overlap)}) -- keeping the TRAIN month's real "
              "aggregate, dropping the duplicate TEST month.")
        _test_monthly = _test_monthly.filter(~pl.col("_month").is_in(_overlap))
    MONTHLY_PORTFOLIO_STORE = pl.concat([_train_monthly, _test_monthly], how="vertical").sort("_month")
else:
    MONTHLY_PORTFOLIO_STORE = _train_monthly

N_CALENDAR_MONTHS_COVERED = MONTHLY_PORTFOLIO_STORE.height
N_LABELED_CALENDAR_MONTHS = _train_monthly.height
print(f"Combined real monthly portfolio store: {N_CALENDAR_MONTHS_COVERED} real calendar months "
      f"({N_LABELED_CALENDAR_MONTHS} labeled/train, "
      f"{N_CALENDAR_MONTHS_COVERED - N_LABELED_CALENDAR_MONTHS} unlabeled/test extension)")
print("\n\u2705 Section 6 complete.")


# =============================================================================
# SECTION 7: BUILD CUSTOMER MONTH-SET & STATEMENT-VALUE STORES -- REUSABLE
#            FUNCTIONS
# =============================================================================
_section("SECTION 7: Build Customer Month-Set & Statement-Value Stores -- Reusable Functions")


def build_customer_month_set_store(csv_path: Path) -> "pl.DataFrame":
    """Streams csv_path and returns one row per customer_ID: the real SET of
    distinct calendar months in which that customer has a real statement.
    REVISED 2026-08-26 (user-directed fix): the original version of this
    function returned only each customer's single LATEST statement month (a
    'cohort month'), tying their whole KPI evaluation to one calendar value
    -- the same 'latest statement' concept Problem 7 uses at the per-customer
    level. In Problem 7 that works because the EVALUATION itself is also
    per-customer (a z-score against that customer's own history). Here the
    ALERT signal is portfolio-level (whole calendar months), and this dataset
    is a fixed snapshot -- so virtually every customer's single latest month
    landed on the SAME shared terminal calendar month, collapsing the entire
    holdout onto one value (confirmed on a real run: 100% of customers
    alerted at every candidate, ROC-AUC exactly 0.5, MCC=0 -- zero real
    customer-level discrimination, an artifact of the single-month design,
    not a real result about the technique). This version instead tracks
    EVERY month a customer was genuinely present, so their ALERT exposure
    can vary with their real month-by-month presence in the portfolio."""
    lf = (
        pl.scan_csv(str(csv_path), schema_overrides={"customer_ID": pl.Utf8, "S_2": pl.Utf8})
        .select(["customer_ID", "S_2"])
        .with_columns(pl.col("S_2").str.to_date("%Y-%m-%d").dt.truncate("1mo").alias("_month"))
        .group_by("customer_ID")
        .agg(pl.col("_month").unique().alias("_months"))
    )
    return lf.collect(engine="streaming")


def build_customer_statement_value_store(csv_path: Path, monitored_cols: list) -> "pl.DataFrame":
    """ADDED 2026-08-27 (user-directed second fix). Streams csv_path and
    returns one row per (customer_ID, real calendar month) with that
    customer's own real raw value for every monitored column that month --
    the individual-level counterpart to build_monthly_portfolio_store's
    AGGREGATE monthly means. Used to test whether a given customer's OWN
    reading was itself an outlier relative to that month's real
    cross-sectional (peer) distribution, the fix for the presence-only
    predicate's saturation (see build_customer_month_set_store's docstring
    and Notebook 58 Section 8 for the full real-run diagnosis). Grouped by
    (customer_ID, month) with .mean() as a defensive aggregate in case a
    customer ever has more than one real statement in the same calendar
    month (not expected for this dataset's monthly cadence, but not
    silently assumed either)."""
    schema_overrides = {"customer_ID": pl.Utf8, "S_2": pl.Utf8}
    for c in monitored_cols:
        schema_overrides[c] = pl.Float32
    _inf_clean_exprs = [
        pl.when(pl.col(c).is_infinite()).then(None).otherwise(pl.col(c)).alias(c)
        for c in monitored_cols
    ]
    lf = (
        pl.scan_csv(str(csv_path), schema_overrides=schema_overrides)
        .select(["customer_ID", "S_2"] + monitored_cols)
        .with_columns(pl.col("S_2").str.to_date("%Y-%m-%d").dt.truncate("1mo").alias("_month"))
        .with_columns(_inf_clean_exprs)
        .group_by(["customer_ID", "_month"])
        .agg([pl.col(c).mean().alias(c) for c in monitored_cols])
    )
    return lf.collect(engine="streaming")


print("build_customer_month_set_store() and build_customer_statement_value_store() defined.")
_t0 = time.time()
cohort_month_store = build_customer_month_set_store(RAW_TRAIN_DATA_PATH)
print(f"Built real customer-month-set store for {cohort_month_store.height:,} train customers in "
      f"{time.time() - _t0:.1f}s. Process RSS: {_rss_gb():.2f} GB")
print("\n\u2705 Section 7 complete.")


# =============================================================================
# SECTION 8: LOAD LABELS & TRAIN/HOLDOUT SPLIT MEMBERSHIP
# =============================================================================
_section("SECTION 8: Load Labels & Train/Holdout Split Membership")

labels_df = pl.read_csv(str(RAW_TRAIN_LABELS_PATH), schema_overrides={"customer_ID": pl.Utf8, "target": pl.Int8})
print(f"Live-read {RAW_TRAIN_LABELS_PATH.name}: {labels_df.shape[0]:,} labeled customers")

val_ids_set = set(pl.read_csv(str(TEST_SPLIT_PATH), columns=["customer_ID"])["customer_ID"].to_list())
print(f"Validation-split customers (from Notebook 02, reused): {len(val_ids_set):,}")

cohort_engineered = cohort_month_store.join(labels_df, on="customer_ID", how="inner")
del cohort_month_store
gc.collect()
cohort_holdout_df = cohort_engineered.filter(pl.col("customer_ID").is_in(val_ids_set))
print(f"Real labeled customers with a real cohort month: {cohort_engineered.height:,} total -- "
      f"{cohort_holdout_df.height:,} holdout-split (evaluated below)")
del cohort_engineered
gc.collect()
print("\n\u2705 Section 8 complete.")


# =============================================================================
# SECTION 9: REAL TRAILING-BASELINE CONTROL-LIMIT COMPUTATION
# =============================================================================
_section("SECTION 9: Real Trailing-Baseline Control-Limit Computation")

print(
    "Per-column TRAILING (fixed-size rolling window) baseline: for real calendar month at index i, the "
    "baseline mean/std is computed from the most recent MIN_TRAILING_MONTHS_FOR_BASELINE real prior months "
    "[i-W, i-1] -- never the month itself, never a future month, never older history than the window. "
    "REVISED 2026-08-26 (user-directed fix): the original version used an EXPANDING (all-history) baseline, "
    "which is the wrong convention for a whole-PORTFOLIO monthly average (unlike Problem 7's per-customer "
    "baseline, where a genuine population-wide secular trend is not expected) -- with real month-over-month "
    "portfolio drift, an all-history baseline falls further and further behind the recent normal, so LATER "
    "months systematically look more 'deviant' purely from drift, not a real shock. Confirmed on a real run: "
    "the resulting ALERT months formed one long 7-month CONSECUTIVE block ending exactly at the window's "
    "final month -- the signature of chronic drift, not episodic anomalies. A genuine trailing window can't "
    "drift this way, since it always compares against a stable-size recent reference. A month needs >= "
    "MIN_TRAILING_MONTHS_FOR_BASELINE real prior months to be evaluated at all; below that, it is honestly "
    "EXCLUDED, not scored against a degenerate baseline."
)

_sorted_store = MONTHLY_PORTFOLIO_STORE.sort("_month")
_months_list = _sorted_store["_month"].to_list()
_n_months = len(_months_list)

_col_series = {c: _sorted_store[f"{c}_portfolio_mean"].to_numpy().astype(np.float64) for c in MONITORED_BASE_COLUMNS}

BASELINE_ELIGIBLE = np.zeros(_n_months, dtype=bool)
BREACH_MATRIX = np.zeros((_n_months, N_MONITORED_COLUMNS), dtype=bool)
Z_SCORE_MATRIX = np.full((_n_months, N_MONITORED_COLUMNS), np.nan, dtype=np.float64)

for i in range(_n_months):
    if i < MIN_TRAILING_MONTHS_FOR_BASELINE:
        continue
    BASELINE_ELIGIBLE[i] = True
    for j, c in enumerate(MONITORED_BASE_COLUMNS):
        _baseline_vals = _col_series[c][max(0, i - MIN_TRAILING_MONTHS_FOR_BASELINE):i]
        _baseline_vals = _baseline_vals[~np.isnan(_baseline_vals)]
        if len(_baseline_vals) < 2:
            continue
        _b_mean = float(np.mean(_baseline_vals))
        _b_std = float(np.std(_baseline_vals, ddof=1))
        _current = _col_series[c][i]
        if _b_std <= 0 or np.isnan(_current):
            continue
        _z = (_current - _b_mean) / _b_std
        Z_SCORE_MATRIX[i, j] = _z
        BREACH_MATRIX[i, j] = bool(abs(_z) >= CONTROL_LIMIT_K_SIGMA)

MONTHLY_BREACH_COUNT = BREACH_MATRIX.sum(axis=1)
N_BASELINE_ELIGIBLE_MONTHS = int(BASELINE_ELIGIBLE.sum())
print(f"Real calendar months in the combined store: {_n_months}")
print(f"Baseline-eligible months (real, >= {MIN_TRAILING_MONTHS_FOR_BASELINE} real prior months): "
      f"{N_BASELINE_ELIGIBLE_MONTHS}")
if N_BASELINE_ELIGIBLE_MONTHS > 0:
    _eligible_breach_counts = MONTHLY_BREACH_COUNT[BASELINE_ELIGIBLE]
    print(f"Real monthly breach-count distribution among eligible months: "
          f"min={_eligible_breach_counts.min()}, median={np.median(_eligible_breach_counts):.1f}, "
          f"max={_eligible_breach_counts.max()}, of {N_MONITORED_COLUMNS} monitored columns")
print("\n\u2705 Section 9 complete.")


# =============================================================================
# SECTION 10: DETERMINE CONSECUTIVE-BREACH ALERT MONTHS PER CANDIDATE
# =============================================================================
_section("SECTION 10: Determine Consecutive-Breach ALERT Months per Candidate")

print(
    "A calendar month is a real 'breaching month' if it is baseline-eligible AND at least one monitored "
    "column's real z-score clears CONTROL_LIMIT_K_SIGMA. For each CONSECUTIVE_BREACH_CANDIDATES "
    "candidate c, a month is in real ALERT state if the run of consecutive breaching months ending at "
    "(and including) it has length >= c -- turning single-month noise into a real, persistence-confirmed "
    "signal before the whole portfolio is declared in an alert state."
)

BREACHING_MONTH = BASELINE_ELIGIBLE & (MONTHLY_BREACH_COUNT >= 1)

ALERT_MONTHS_BY_CANDIDATE = {}
for _cand in CONSECUTIVE_BREACH_CANDIDATES:
    _alert = np.zeros(_n_months, dtype=bool)
    _run_len = 0
    for i in range(_n_months):
        if BREACHING_MONTH[i]:
            _run_len += 1
        else:
            _run_len = 0
        _alert[i] = _run_len >= _cand
    ALERT_MONTHS_BY_CANDIDATE[int(_cand)] = _alert
    _n_alert_months = int(_alert.sum())
    print(f"  candidate={_cand}: {_n_alert_months} of {_n_months} real calendar months in ALERT state "
          f"({[str(m) for m, a in zip(_months_list, _alert) if a]})")
print("\n\u2705 Section 10 complete.")


# =============================================================================
# SECTION 11: COHORT DEFAULT-RATE LIFT + FULL METRICS SUITE PER CANDIDATE
# =============================================================================
_section("SECTION 11: Cohort Default-Rate Lift + Full Metrics Suite per Candidate")

# --- KPI evaluation population: real holdout-split customers whose real
#     cohort month is BASELINE-ELIGIBLE (a customer whose only statements
#     fall in a month too early to have a real baseline could never be
#     flagged either way -- honestly excluded from this KPI evaluation
#     population, exactly as Problem 7 excludes customers below its own
#     baseline-eligibility bar). ---
_eligible_months_set = {m for m, e in zip(_months_list, BASELINE_ELIGIBLE) if e}
_month_to_idx = {m: i for i, m in enumerate(_months_list)}

# --- KPI evaluation population: real holdout-split customers with at least
#     ONE real statement in a baseline-eligible month (not just whose single
#     LATEST statement happens to be eligible -- see build_customer_month_
#     set_store's docstring for why that collapsed the population). ---
_cohort_eval_df = cohort_holdout_df.with_columns(
    pl.col("_months").list.set_intersection(list(_eligible_months_set)).alias("_eligible_present_months")
).filter(pl.col("_eligible_present_months").list.len() > 0)
N_HOLDOUT_EVAL = _cohort_eval_df.height
print(f"Real holdout customers with >=1 real statement in a baseline-eligible month (KPI evaluation "
      f"population): {N_HOLDOUT_EVAL:,} of {cohort_holdout_df.height:,} real holdout customers")

if N_HOLDOUT_EVAL == 0:
    print("NOTE: zero real holdout customers have a real statement in a baseline-eligible month -- no KPI "
          "evaluation is possible against the real calendar coverage this run found. Reported honestly; "
          "Notebook 60 must NOT RECOMMEND FOR PRODUCTION rather than fabricate a result.")
    y_eval = np.array([], dtype=np.int64)
    PRESENCE = np.zeros((0, _n_months), dtype=bool)
    JOINT_CANDIDATE_MASK = np.zeros((0, _n_months, N_MONITORED_COLUMNS), dtype=bool)
    CUSTOMER_JOINT_DEVIATION_COUNT = np.array([], dtype=np.float64)
else:
    y_eval = _cohort_eval_df.get_column("target").to_numpy().astype(np.int64)
    _present_months_list = _cohort_eval_df.get_column("_eligible_present_months").to_list()
    # PRESENCE[i, j] = True iff eval customer i has a real statement in real calendar month j.
    PRESENCE = np.zeros((N_HOLDOUT_EVAL, _n_months), dtype=bool)
    for _i, _months_for_cust in enumerate(_present_months_list):
        for _m in _months_for_cust:
            PRESENCE[_i, _month_to_idx[_m]] = True

    # =========================================================================
    # SECTION 11B: REAL PER-CUSTOMER, PER-MONTH, PER-COLUMN JOINT DEVIATION
    #              (ADDED 2026-08-27, user-directed second real-data fix)
    # =========================================================================
    print(
        "\nADDED 2026-08-27 (second real-data fix): the presence-only predicate ('was this customer active "
        "during ANY real ALERT month') gave >=75% of customers -- everyone with full 13/13-month coverage "
        "-- virtually identical exposure, confirmed on real runs as exactly 100%/0% alerted with zero real "
        "discrimination. FIX: a customer is now only counted if, in a real month/column the PORTFOLIO "
        "itself breached, that customer's OWN real statement value in that SAME column ALSO sits >= "
        "CUSTOMER_DEVIATION_Z_THRESHOLD real cross-sectional standard deviations from that month's real "
        "peer mean, in the SAME direction as the portfolio's own shift -- was this customer part of what "
        "actually drove the anomaly, not merely present while it happened. This restores real per-customer "
        "variation even within the fully-covered majority, since their real raw values differ month to "
        "month even when their real month-presence does not."
    )

    _t0 = time.time()
    _eval_customer_ids = _cohort_eval_df.get_column("customer_ID").to_list()
    _cust_id_to_idx = {cid: i for i, cid in enumerate(_eval_customer_ids)}
    _statement_value_store = build_customer_statement_value_store(RAW_TRAIN_DATA_PATH, MONITORED_BASE_COLUMNS)
    _eval_ids_df = pl.DataFrame({"customer_ID": _eval_customer_ids})
    _eval_values_df = _statement_value_store.join(_eval_ids_df, on="customer_ID", how="inner")
    del _statement_value_store
    gc.collect()
    print(f"Built real per-customer statement-value store: {_eval_values_df.height:,} real (customer, month) "
          f"rows for {N_HOLDOUT_EVAL:,} evaluation customers in {time.time() - _t0:.1f}s. "
          f"Process RSS: {_rss_gb():.2f} GB")

    # CUSTOMER_VALUES[i, m, j] = eval customer i's real raw value for monitored column j in real calendar
    # month m (NaN where that customer has no real statement that month).
    CUSTOMER_VALUES = np.full((N_HOLDOUT_EVAL, _n_months, N_MONITORED_COLUMNS), np.nan, dtype=np.float32)
    _row_cust_idx = np.array([_cust_id_to_idx[c] for c in _eval_values_df.get_column("customer_ID").to_list()])
    _row_month_idx = np.array([_month_to_idx[m] for m in _eval_values_df.get_column("_month").to_list()])
    for _j, _c in enumerate(MONITORED_BASE_COLUMNS):
        CUSTOMER_VALUES[_row_cust_idx, _row_month_idx, _j] = _eval_values_df.get_column(_c).to_numpy().astype(np.float32)
    del _eval_values_df
    gc.collect()

    # CROSS_SECTIONAL_MEAN/STD[m, j] = real peer (cross-sectional) mean/std of monitored column j among ALL
    # customers with a real statement in real calendar month m -- the right-scale reference for judging ONE
    # customer's own reading (unlike the trailing-baseline-of-MEANS mean/std Section 9 uses for the
    # portfolio-wide aggregate, which has much lower variance and is the wrong reference for an individual).
    CROSS_SECTIONAL_MEAN = np.stack(
        [_sorted_store[f"{c}_portfolio_mean"].to_numpy().astype(np.float32) for c in MONITORED_BASE_COLUMNS], axis=1
    )
    CROSS_SECTIONAL_STD = np.stack(
        [_sorted_store[f"{c}_portfolio_std"].to_numpy().astype(np.float32) for c in MONITORED_BASE_COLUMNS], axis=1
    )

    with np.errstate(invalid="ignore", divide="ignore"):
        CUSTOMER_Z = (CUSTOMER_VALUES - CROSS_SECTIONAL_MEAN[np.newaxis, :, :]) / CROSS_SECTIONAL_STD[np.newaxis, :, :]
    _z_computable = (
        ~np.isnan(CUSTOMER_VALUES)
        & ~np.isnan(CROSS_SECTIONAL_MEAN)[np.newaxis, :, :]
        & (CROSS_SECTIONAL_STD[np.newaxis, :, :] > 0)
    )
    CUSTOMER_Z = np.where(_z_computable, CUSTOMER_Z, np.nan)

    with np.errstate(invalid="ignore"):
        _portfolio_z_sign = np.sign(Z_SCORE_MATRIX)          # shape (n_months, n_cols); NaN where not computable
        _customer_z_sign = np.sign(CUSTOMER_Z)                # shape (N_HOLDOUT_EVAL, n_months, n_cols)
        # JOINT_CANDIDATE_MASK[i, m, j]: True iff (1) the PORTFOLIO itself really breached column j in month
        # m (BREACH_MATRIX), (2) eval customer i has a real, z-computable value there, (3) that customer's
        # own real cross-sectional deviation clears CUSTOMER_DEVIATION_Z_THRESHOLD, and (4) it moved in the
        # SAME real direction as the portfolio's own shift that month/column.
        JOINT_CANDIDATE_MASK = (
            _z_computable
            & (np.abs(CUSTOMER_Z) >= CUSTOMER_DEVIATION_Z_THRESHOLD)
            & (_customer_z_sign == _portfolio_z_sign[np.newaxis, :, :])
            & BREACH_MATRIX[np.newaxis, :, :]
        )
    CUSTOMER_JOINT_DEVIATION_COUNT = JOINT_CANDIDATE_MASK.sum(axis=(1, 2)).astype(np.float64)
    print(f"Real customer-level joint-deviation count distribution (real, measured, across all "
          f"{N_MONITORED_COLUMNS} monitored columns x {N_BASELINE_ELIGIBLE_MONTHS} baseline-eligible "
          f"months): min={CUSTOMER_JOINT_DEVIATION_COUNT.min():.0f}, "
          f"median={np.median(CUSTOMER_JOINT_DEVIATION_COUNT):.1f}, "
          f"mean={CUSTOMER_JOINT_DEVIATION_COUNT.mean():.3f}, max={CUSTOMER_JOINT_DEVIATION_COUNT.max():.0f}")
    print(f"Real customers with >=1 real joint deviation (any month/column): "
          f"{int((CUSTOMER_JOINT_DEVIATION_COUNT > 0).sum()):,} of {N_HOLDOUT_EVAL:,} "
          f"({100.0 * (CUSTOMER_JOINT_DEVIATION_COUNT > 0).mean():.2f}%)")

BASE_DEFAULT_RATE_HOLDOUT = float(y_eval.mean()) if N_HOLDOUT_EVAL > 0 else None
print(f"\nBase holdout default rate among the eligible evaluation population (real, measured): "
      f"{BASE_DEFAULT_RATE_HOLDOUT}")

PORTFOLIO_MODELING_RESULTS = {}
if N_HOLDOUT_EVAL > 0:
    print(f"\n{'candidate':>9} {'n_alert':>8} {'%alert':>7} {'default%':>9} {'lift':>6} {'KPI':>8} "
          f"{'acc':>6} {'prec':>6} {'rec':>6} {'f1':>6} {'spec':>6} {'mcc':>6}  confusion(tn,fp,fn,tp)")
    for _cand in CONSECUTIVE_BREACH_CANDIDATES:
        _alert_arr = ALERT_MONTHS_BY_CANDIDATE[int(_cand)]
        # A customer is ALERTED if they have >=1 real JOINT deviation (portfolio breach in a column AND
        # their own reading in that same column/month is itself an outlier in the same direction) that
        # falls in one of this candidate's real, persistence-confirmed ALERT months -- not just presence.
        pred = JOINT_CANDIDATE_MASK[:, _alert_arr, :].any(axis=(1, 2)).astype(np.int64)
        tn, fp, fn, tp = confusion_matrix(y_eval, pred, labels=[0, 1]).ravel()
        specificity = float(tn / (tn + fp)) if (tn + fp) > 0 else 0.0
        n_alerted = int(pred.sum())
        default_rate_alerted = float(y_eval[pred == 1].mean()) if n_alerted > 0 else None
        lift = (
            (default_rate_alerted / BASE_DEFAULT_RATE_HOLDOUT)
            if (default_rate_alerted is not None and BASE_DEFAULT_RATE_HOLDOUT > 0) else None
        )
        meets_kpi = bool(lift is not None and lift >= PORTFOLIO_KPI_TARGETS["min_cohort_default_rate_lift"])

        metrics = {
            "candidate_consecutive_breach_months": int(_cand),
            "n_alerted": n_alerted,
            "pct_alerted": 100.0 * n_alerted / N_HOLDOUT_EVAL,
            "default_rate_alerted": default_rate_alerted,
            "base_default_rate_holdout": BASE_DEFAULT_RATE_HOLDOUT,
            "default_rate_lift": lift,
            "meets_kpi_target": meets_kpi,
            "accuracy": float(accuracy_score(y_eval, pred)),
            "precision": float(precision_score(y_eval, pred, zero_division=0)),
            "recall": float(recall_score(y_eval, pred, zero_division=0)),
            "f1": float(f1_score(y_eval, pred, zero_division=0)),
            "specificity": specificity,
            "mcc": float(matthews_corrcoef(y_eval, pred)),
            "confusion_matrix": {"tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp)},
        }
        PORTFOLIO_MODELING_RESULTS[int(_cand)] = metrics

        _default_pct_str = f"{default_rate_alerted*100:.2f}" if default_rate_alerted is not None else "n/a"
        _lift_str = f"{lift:.2f}x" if lift is not None else "n/a"
        print(f"{_cand:>9} {n_alerted:>8,} {metrics['pct_alerted']:>6.2f}% {_default_pct_str:>8}% {_lift_str:>6} "
              f"{('MET' if meets_kpi else 'NOT MET'):>8} "
              f"{metrics['accuracy']:>6.4f} {metrics['precision']:>6.4f} {metrics['recall']:>6.4f} "
              f"{metrics['f1']:>6.4f} {metrics['specificity']:>6.4f} {metrics['mcc']:>6.4f}  "
              f"({tn:,}, {fp:,}, {fn:,}, {tp:,})")

_candidates_meeting_kpi = [c for c, m in PORTFOLIO_MODELING_RESULTS.items() if m["meets_kpi_target"]]
_meeting_kpi_str = (
    str(_candidates_meeting_kpi) if _candidates_meeting_kpi
    else "NONE -- reported plainly, see Notebook 60 for the honest winning-candidate selection."
)
print(f"\nCandidate(s) meeting the >= {PORTFOLIO_KPI_TARGETS['min_cohort_default_rate_lift']}x lift KPI: "
      f"{_meeting_kpi_str}")
print("\n\u2705 Section 11 complete.")


# =============================================================================
# SECTION 12: THRESHOLD-FREE SECONDARY EVALUATION -- CONTINUOUS CUSTOMER
#             JOINT-DEVIATION SCORE (SECONDARY, NON-GATING REQUIREMENT)
# =============================================================================
_section("SECTION 12: Threshold-Free Secondary Evaluation -- Continuous Customer Joint-Deviation Score")

if N_HOLDOUT_EVAL > 0:
    # REVISED 2026-08-27: normalized real per-customer JOINT deviation count (portfolio breach + own
    # cross-sectional outlier, same direction -- see Section 11B) replaces the old presence-only
    # CUSTOMER_MAX_BREACH_COUNT score, for the same reason the primary KPI predicate changed.
    _cohort_score_normalized = CUSTOMER_JOINT_DEVIATION_COUNT / max(1, N_MONITORED_COLUMNS * N_BASELINE_ELIGIBLE_MONTHS)

    holdout_auc = float(roc_auc_score(y_eval, _cohort_score_normalized)) if len(set(y_eval.tolist())) > 1 else None
    holdout_pr_auc = float(average_precision_score(y_eval, _cohort_score_normalized)) if holdout_auc is not None else None
    if holdout_auc is not None:
        _eps = 1e-7
        _clipped_score = np.clip(_cohort_score_normalized, _eps, 1.0 - _eps)
        holdout_log_loss = float(log_loss(y_eval, _clipped_score, labels=[0, 1]))
        fpr, tpr, _roc_thresholds = roc_curve(y_eval, _cohort_score_normalized)
        pr_precision, pr_recall, _pr_thresholds = precision_recall_curve(y_eval, _cohort_score_normalized)
    else:
        holdout_log_loss = None
        fpr = tpr = pr_precision = pr_recall = np.array([])
        print("NOTE: the real evaluation population contains only one class (all-default or all-non-default) "
              "-- ROC-AUC/PR-AUC are honestly undefined, reported as null rather than fabricated.")

    auc_retention_pct = (
        float(holdout_auc / FULL_HISTORY_AUC * 100.0) if (holdout_auc is not None and FULL_HISTORY_AUC) else None
    )
    print(f"Continuous customer joint-deviation score ROC-AUC vs. real label: {holdout_auc}")
    print(f"Continuous customer joint-deviation score PR-AUC vs. real label : {holdout_pr_auc}")
    print(f"Continuous customer joint-deviation score Log Loss              : {holdout_log_loss}")
    if auc_retention_pct is not None:
        print(f"(Informational only, not a KPI here) Retains {auc_retention_pct:.1f}% of Problem 1's "
              f"full-history champion AUC ({FULL_HISTORY_AUC:.4f}) -- an honestly low figure is the "
              "EXPECTED outcome for a coarse, month-level portfolio aggregate signal, not a failure.")
else:
    holdout_auc = holdout_pr_auc = holdout_log_loss = auc_retention_pct = None
    fpr = tpr = pr_precision = pr_recall = np.array([])
print("\n\u2705 Section 12 complete.")


# =============================================================================
# SECTION 13: CHARTS -- MONTHLY KPI TREND, LIFT BY CANDIDATE, ROC/PR CURVES
# =============================================================================
_section("SECTION 13: Charts -- Monthly KPI Trend, Lift by Candidate, ROC/PR Curves")

CHARTS_DIR = P11_REPORTS_DIR / "charts"
CHARTS_DIR.mkdir(parents=True, exist_ok=True)

_headline_col = MONITORED_BASE_COLUMNS[0]
_headline_vals = _col_series[_headline_col]
_month_labels = [str(m) for m in _months_list]

fig0, ax0 = plt.subplots(figsize=(11, 5))
ax0.plot(_month_labels, _headline_vals, color="#1f77b4", lw=2, marker="o", markersize=4,
         label=f"{_headline_col} portfolio mean")
_breach_x = [m for m, b in zip(_month_labels, BREACH_MATRIX[:, 0]) if b]
_breach_y = [v for v, b in zip(_headline_vals, BREACH_MATRIX[:, 0]) if b]
if _breach_x:
    ax0.scatter(_breach_x, _breach_y, color="#d62728", s=70, zorder=5, label="Real control-limit breach")
ax0.set_xlabel("Real calendar month")
ax0.set_ylabel(f"{_headline_col} (portfolio mean)")
ax0.set_title(f"Problem 11: Real Monthly Portfolio Trend -- {_headline_col}\n"
              f"(top SHAP/importance-ranked monitored column, real breaches marked)", fontsize=11)
ax0.tick_params(axis="x", rotation=60)
ax0.legend(loc="best")
ax0.grid(alpha=0.3)
fig0.tight_layout()
chart0_path = CHARTS_DIR / "notebook_59_monthly_kpi_trend.png"
fig0.savefig(chart0_path, dpi=150)
plt.close(fig0)
print(f"Saved: {chart0_path}")

fig3, ax3 = plt.subplots(figsize=(7, 5))
if PORTFOLIO_MODELING_RESULTS:
    _cands_sorted = sorted(PORTFOLIO_MODELING_RESULTS.keys())
    _lifts = [PORTFOLIO_MODELING_RESULTS[c]["default_rate_lift"] or 0.0 for c in _cands_sorted]
    _colors = ["#2ca02c" if PORTFOLIO_MODELING_RESULTS[c]["meets_kpi_target"] else "#7f7f7f" for c in _cands_sorted]
    ax3.bar([str(c) for c in _cands_sorted], _lifts, color=_colors)
    ax3.axhline(PORTFOLIO_KPI_TARGETS["min_cohort_default_rate_lift"], color="#d62728", lw=1.5, linestyle="--",
                label=f"KPI target ({PORTFOLIO_KPI_TARGETS['min_cohort_default_rate_lift']}x)")
    ax3.legend(loc="best")
else:
    ax3.text(0.5, 0.5, "No KPI-evaluable candidates\n(zero baseline-eligible holdout cohorts)",
              ha="center", va="center", transform=ax3.transAxes)
ax3.set_xlabel("CONSECUTIVE_BREACH_CANDIDATES candidate")
ax3.set_ylabel("Default-rate lift (alerted cohort vs. base population)")
ax3.set_title("Problem 11: Real Cohort Default-Rate Lift by Consecutive-Breach Candidate\n"
              "(green = meets KPI, gray = does not)", fontsize=11)
ax3.grid(alpha=0.3, axis="y")
fig3.tight_layout()
chart3_path = CHARTS_DIR / "notebook_59_lift_by_candidate.png"
fig3.savefig(chart3_path, dpi=150)
plt.close(fig3)
print(f"Saved: {chart3_path}")

fig1, ax1 = plt.subplots(figsize=(7, 6))
if holdout_auc is not None:
    ax1.plot(fpr, tpr, color="#1f77b4", lw=2, label=f"Customer joint-deviation score (AUC = {holdout_auc:.4f})")
    ax1.plot([0, 1], [0, 1], color="gray", lw=1, linestyle="--", label="Random (AUC = 0.500)")
    ax1.legend(loc="lower right")
else:
    ax1.text(0.5, 0.5, "ROC-AUC undefined\n(single-class evaluation population)", ha="center", va="center",
              transform=ax1.transAxes)
ax1.set_xlabel("False Positive Rate")
ax1.set_ylabel("True Positive Rate")
ax1.set_title("Problem 11: ROC Curve -- Customer Joint-Deviation Score\n(real, holdout-split cohorts)", fontsize=11)
ax1.grid(alpha=0.3)
fig1.tight_layout()
chart1_path = CHARTS_DIR / "notebook_59_roc_curve.png"
fig1.savefig(chart1_path, dpi=150)
plt.close(fig1)
print(f"Saved: {chart1_path}")

fig2, ax2 = plt.subplots(figsize=(7, 6))
if holdout_pr_auc is not None:
    ax2.plot(pr_recall, pr_precision, color="#d62728", lw=2, label=f"Customer joint-deviation score (PR-AUC = {holdout_pr_auc:.4f})")
    if BASE_DEFAULT_RATE_HOLDOUT is not None:
        ax2.axhline(BASE_DEFAULT_RATE_HOLDOUT, color="gray", lw=1, linestyle="--",
                    label=f"Base rate ({BASE_DEFAULT_RATE_HOLDOUT:.3f})")
    ax2.legend(loc="upper right")
else:
    ax2.text(0.5, 0.5, "PR-AUC undefined\n(single-class evaluation population)", ha="center", va="center",
              transform=ax2.transAxes)
ax2.set_xlabel("Recall")
ax2.set_ylabel("Precision")
ax2.set_title("Problem 11: Precision-Recall Curve -- Customer Joint-Deviation Score\n(real, holdout-split cohorts)",
              fontsize=11)
ax2.grid(alpha=0.3)
fig2.tight_layout()
chart2_path = CHARTS_DIR / "notebook_59_pr_curve.png"
fig2.savefig(chart2_path, dpi=150)
plt.close(fig2)
print(f"Saved: {chart2_path}")
print("\n\u2705 Section 13 complete.")


# =============================================================================
# SECTION 14: WRITE MODELING RESULTS ARTIFACT
# =============================================================================
_section("SECTION 14: Write Modeling Results Artifact")

MODELING_RESULTS_ARTIFACT = {
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "problem": "Problem 11 -- Real-Time Portfolio Monitoring (Streaming Aggregation + Threshold Alerting)",
    "control_limit_k_sigma": CONTROL_LIMIT_K_SIGMA,
    "min_trailing_months_for_baseline": MIN_TRAILING_MONTHS_FOR_BASELINE,
    "monitored_base_columns": MONITORED_BASE_COLUMNS,
    "n_calendar_months_covered": _n_months,
    "n_labeled_calendar_months": N_LABELED_CALENDAR_MONTHS,
    "n_baseline_eligible_months": N_BASELINE_ELIGIBLE_MONTHS,
    "n_holdout_eval_customers": N_HOLDOUT_EVAL,
    "base_default_rate_holdout": BASE_DEFAULT_RATE_HOLDOUT,
    "monthly_breach_count_distribution": (
        {
            "min": int(MONTHLY_BREACH_COUNT[BASELINE_ELIGIBLE].min()),
            "median": float(np.median(MONTHLY_BREACH_COUNT[BASELINE_ELIGIBLE])),
            "mean": float(MONTHLY_BREACH_COUNT[BASELINE_ELIGIBLE].mean()),
            "max": int(MONTHLY_BREACH_COUNT[BASELINE_ELIGIBLE].max()),
        } if N_BASELINE_ELIGIBLE_MONTHS > 0 else None
    ),
    "secondary_threshold_free_metrics": {
        "roc_auc": holdout_auc,
        "pr_auc": holdout_pr_auc,
        "log_loss": holdout_log_loss,
        "auc_retention_pct_of_full_history": auc_retention_pct,
        "full_history_reference_auc": FULL_HISTORY_AUC,
    },
    "candidate_results": PORTFOLIO_MODELING_RESULTS,
    "candidates_meeting_kpi": _candidates_meeting_kpi,
    "alert_months_by_candidate": {
        str(c): [str(m) for m, a in zip(_months_list, arr) if a] for c, arr in ALERT_MONTHS_BY_CANDIDATE.items()
    },
    "chart_paths": {
        "monthly_kpi_trend": str(chart0_path),
        "roc_curve": str(chart1_path),
        "pr_curve": str(chart2_path),
        "lift_by_candidate": str(chart3_path),
    },
    "random_seed": RANDOM_SEED,
}
modeling_results_path = P11_MODELS_DIR / "portfolio_monitoring_modeling_results.json"
with open(modeling_results_path, "w", encoding="utf-8") as f:
    json.dump(MODELING_RESULTS_ARTIFACT, f, indent=2)
print(f"Wrote: {modeling_results_path}")
print("\n\u2705 Section 14 complete.")


# =============================================================================
# SECTION 15: VERIFICATION -- INTEGRITY CHECKS
# =============================================================================
_section("SECTION 15: Verification -- Integrity Checks")


def _check(label, condition, detail=""):
    status = "PASS" if condition else "FAIL"
    print(f"  [{status}] {label}" + (f" -- {detail}" if detail and not condition else ""))
    return condition


_all_checks_passed = True
_all_checks_passed &= _check("Modeling results file was written", modeling_results_path.exists())
_all_checks_passed &= _check("Real calendar-month count is positive", _n_months > 0)
_all_checks_passed &= _check("Baseline-eligible month count matches Section 9's real computation",
                              N_BASELINE_ELIGIBLE_MONTHS == int(BASELINE_ELIGIBLE.sum()))
_all_checks_passed &= _check("Every candidate result key matches the real policy's candidates",
                              set(PORTFOLIO_MODELING_RESULTS.keys()) <= set(int(c) for c in CONSECUTIVE_BREACH_CANDIDATES))
if N_HOLDOUT_EVAL > 0:
    _all_checks_passed &= _check("Every candidate in the policy has a result (non-empty evaluation population)",
                                  set(PORTFOLIO_MODELING_RESULTS.keys()) == set(int(c) for c in CONSECUTIVE_BREACH_CANDIDATES))
    _all_checks_passed &= _check(
        "Every candidate's confusion matrix sums to the real evaluation-population count",
        all(sum(PORTFOLIO_MODELING_RESULTS[c]["confusion_matrix"].values()) == N_HOLDOUT_EVAL
            for c in PORTFOLIO_MODELING_RESULTS)
    )
    _cands_sorted_check = sorted(PORTFOLIO_MODELING_RESULTS.keys())
    _all_checks_passed &= _check(
        "pct_alerted is non-increasing as the consecutive-breach candidate rises (a stricter bar alerts no more)",
        all(PORTFOLIO_MODELING_RESULTS[_cands_sorted_check[i]]["pct_alerted"]
            >= PORTFOLIO_MODELING_RESULTS[_cands_sorted_check[i + 1]]["pct_alerted"]
            for i in range(len(_cands_sorted_check) - 1))
    )
    if holdout_auc is not None:
        _all_checks_passed &= _check("ROC-AUC is a real probability-like value in [0, 1]", 0.0 <= holdout_auc <= 1.0)
        _all_checks_passed &= _check("PR-AUC is a real probability-like value in [0, 1]", 0.0 <= holdout_pr_auc <= 1.0)
_all_checks_passed &= _check("MONTHLY_BREACH_COUNT never exceeds the monitored-column count",
                              int(MONTHLY_BREACH_COUNT.max()) <= N_MONITORED_COLUMNS if _n_months > 0 else True)
_all_checks_passed &= _check("All four chart PNGs were written",
                              all(p.exists() for p in [chart0_path, chart1_path, chart2_path, chart3_path]))
_all_checks_passed &= _check("Reused Problem 11's real policy CONTROL_LIMIT_K_SIGMA/candidates (no re-guessing)",
                              CONSECUTIVE_BREACH_CANDIDATES == PORTFOLIO_MONITORING_POLICY["consecutive_breach_candidates"])
_all_checks_passed &= _check("Reused Problem 11's real policy CUSTOMER_DEVIATION_Z_THRESHOLD (no re-guessing)",
                              CUSTOMER_DEVIATION_Z_THRESHOLD == PORTFOLIO_MONITORING_POLICY["customer_deviation_z_threshold"])
if N_HOLDOUT_EVAL > 0:
    _all_checks_passed &= _check("CUSTOMER_JOINT_DEVIATION_COUNT is real, finite, and non-negative",
                                  bool(np.all(np.isfinite(CUSTOMER_JOINT_DEVIATION_COUNT))
                                       and np.all(CUSTOMER_JOINT_DEVIATION_COUNT >= 0)))
    _all_checks_passed &= _check("JOINT_CANDIDATE_MASK only flags (month, column) pairs the portfolio itself "
                                  "really breached (a real subset of BREACH_MATRIX)",
                                  bool(np.all(JOINT_CANDIDATE_MASK <= BREACH_MATRIX[np.newaxis, :, :])))
_all_checks_passed &= _check("ALERT_MONTHS_BY_CANDIDATE is monotonic: a higher candidate never alerts more months "
                              "than a lower one",
                              all(ALERT_MONTHS_BY_CANDIDATE[c1].sum() >= ALERT_MONTHS_BY_CANDIDATE[c2].sum()
                                  for c1, c2 in zip(sorted(ALERT_MONTHS_BY_CANDIDATE)[:-1], sorted(ALERT_MONTHS_BY_CANDIDATE)[1:])))

if not _all_checks_passed:
    raise AssertionError("One or more verification checks failed -- see FAIL lines above.")
print("\n\u2705 Section 15 complete -- all checks passed.")


# =============================================================================
# SECTION 16: WRITE NOTEBOOK 59 SUMMARY ARTIFACT & COMPLETION
# =============================================================================
_section("SECTION 16: Write Notebook 59 Summary Artifact")

NB59_SUMMARY = {
    "notebook": "59_real_time_portfolio_monitoring_modeling.ipynb",
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "modeling_results_path": str(modeling_results_path),
    "n_calendar_months_covered": _n_months,
    "n_baseline_eligible_months": N_BASELINE_ELIGIBLE_MONTHS,
    "n_holdout_eval_customers": N_HOLDOUT_EVAL,
    "base_default_rate_holdout": BASE_DEFAULT_RATE_HOLDOUT,
    "secondary_roc_auc": holdout_auc,
    "secondary_pr_auc": holdout_pr_auc,
    "candidates_meeting_kpi": _candidates_meeting_kpi,
    "consecutive_breach_candidates": CONSECUTIVE_BREACH_CANDIDATES,
    "chart_paths": {
        "monthly_kpi_trend": str(chart0_path),
        "roc_curve": str(chart1_path),
        "pr_curve": str(chart2_path),
        "lift_by_candidate": str(chart3_path),
    },
    "random_seed": RANDOM_SEED,
}
NB59_SUMMARY_PATH = ARTIFACTS_DIR / "notebook_59_summary.json"
with open(NB59_SUMMARY_PATH, "w", encoding="utf-8") as f:
    json.dump(NB59_SUMMARY, f, indent=2)
print(f"Wrote: {NB59_SUMMARY_PATH}")

_section("NOTEBOOK 59 COMPLETE")
print(f"Real calendar months covered (train{'+ test extension' if _HAS_TEST_DATA else ''}): {_n_months}")
print(f"Baseline-eligible months (real)              : {N_BASELINE_ELIGIBLE_MONTHS}")
print(f"Holdout evaluation population (real)          : {N_HOLDOUT_EVAL:,}")
print(f"Candidate(s) meeting the lift KPI              : {_meeting_kpi_str}")
print(f"Secondary ROC-AUC / PR-AUC                     : {holdout_auc} / {holdout_pr_auc}")
print(f"Modeling results written to: {modeling_results_path}")
print(
    "\nNext: 60_real_time_portfolio_monitoring_validation_deployment.ipynb -- independently reproduces "
    "this notebook's monthly portfolio store and control-chart logic from scratch, cross-checks it "
    "against this notebook's persisted results, bootstraps confidence intervals on the winning "
    "candidate's real cohort default-rate lift, selects the honest winning CONSECUTIVE_BREACH_CANDIDATES "
    "value (or flags NOT RECOMMENDED FOR PRODUCTION), and generates the real ops dashboard + alert-feed "
    "deployment service this problem's deliverable requires."
)

